# QualiSpeech Candidate OOD Test Demo

Builds candidate out-of-domain test clips from QualiSpeech labels and visualizes each clip with:
- waveform + annotated spans
- inline audio playback


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display


In [ ]:
DATA_ROOT = Path("../data/raw/QualiSpeech")
SPLIT = "test"
CSV_PATH = DATA_ROOT / f"{SPLIT}.csv"
WAV_DIR = DATA_ROOT / "wav" / SPLIT

N_PER_GROUP = 4
LOCALIZED_MAX_SECONDS = 2.0

df = pd.read_csv(CSV_PATH)

def extract_spans(text: object) -> list[dict[str, float]]:
    """Extract [start, end] spans from free-text time annotations."""
    if pd.isna(text):
        return []

    value = str(text).strip().lower()
    if not value or "not noticeable" in value or "not distorted" in value or "very smooth" in value:
        return []

    pattern = r"((?:\d+\.\d+)|\d+)\s*(?:-|~|to)\s*((?:\d+\.\d+)|\d+)s?"
    spans: list[dict[str, float]] = []
    for start_str, end_str in re.findall(pattern, value):
        start, end = float(start_str), float(end_str)
        if end < start:
            start, end = end, start
        spans.append({"start": start, "end": end})
    return spans

def span_total_seconds(spans: list[dict[str, float]]) -> float:
    """Compute total duration of spans in seconds."""
    return float(sum(max(0.0, seg["end"] - seg["start"]) for seg in spans))

def starts_near_zero(spans: list[dict[str, float]], atol: float = 0.05) -> bool:
    """Return whether the first span starts near the clip start."""
    if not spans:
        return False
    return min(seg["start"] for seg in spans) <= atol

def pick_examples(frame: pd.DataFrame, group: str, n: int) -> pd.DataFrame:
    """Sample up to n rows and attach a candidate group name."""
    if frame.empty:
        return frame.assign(group=group).head(0)
    out = frame.sample(n=min(n, len(frame)), random_state=7).copy()
    out["group"] = group
    return out

df["wav_path"] = df["id"].apply(lambda file_id: WAV_DIR / file_id)
df = df[df["wav_path"].apply(lambda path: path.exists())].copy()

df["noise_spans"] = df["Noise Description"].apply(extract_spans)
df["distortion_spans"] = df["Distortion description"].apply(extract_spans)
df["pause_spans"] = df["Unnatural pause"].apply(extract_spans)

df["noise_len"] = df["noise_spans"].apply(span_total_seconds)
df["dist_len"] = df["distortion_spans"].apply(span_total_seconds)
df["pause_len"] = df["pause_spans"].apply(span_total_seconds)
df["dist_start_zero"] = df["distortion_spans"].apply(starts_near_zero)

clean_df = df[
    (df["noise_spans"].apply(len) == 0)
    & (df["distortion_spans"].apply(len) == 0)
    & (df["pause_spans"].apply(len) == 0)
]

localized_dist_df = df[
    (df["distortion_spans"].apply(len) > 0)
    & (df["dist_len"] <= LOCALIZED_MAX_SECONDS)
    & (~df["dist_start_zero"])
]

pause_only_df = df[
    (df["pause_spans"].apply(len) > 0)
    & (df["noise_spans"].apply(len) == 0)
    & (df["distortion_spans"].apply(len) == 0)
]

noise_only_local_df = df[
    (df["noise_spans"].apply(len) > 0)
    & (df["distortion_spans"].apply(len) == 0)
    & (df["noise_len"] <= LOCALIZED_MAX_SECONDS)
]

candidate_df = pd.concat(
    [
        pick_examples(clean_df, "clean_baseline", N_PER_GROUP),
        pick_examples(localized_dist_df, "localized_distortion", N_PER_GROUP),
        pick_examples(pause_only_df, "pause_only", N_PER_GROUP),
        pick_examples(noise_only_local_df, "localized_noise_only", N_PER_GROUP),
    ],
    ignore_index=True,
)

summary_df = pd.DataFrame(
    [
        {"group": "clean_baseline", "available": len(clean_df)},
        {"group": "localized_distortion", "available": len(localized_dist_df)},
        {"group": "pause_only", "available": len(pause_only_df)},
        {"group": "localized_noise_only", "available": len(noise_only_local_df)},
    ]
)

display(summary_df)
display(candidate_df[["group", "id", "Overall quality", "Noise Description", "Distortion description", "Unnatural pause"]])
print(f"Loaded split={SPLIT} with {len(df)} clips that have local audio files.")
print(f"Prepared {len(candidate_df)} candidate demo clips ({N_PER_GROUP} per group when available).")


In [ ]:
for _, row in candidate_df.sort_values(["group", "id"]).iterrows():
    wav_path = row["wav_path"]
    audio, sr = sf.read(wav_path)

    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    times = np.arange(len(audio)) / sr
    fig, ax = plt.subplots(figsize=(12, 3.2))

    for seg in row["noise_spans"]:
        ax.axvspan(seg["start"], seg["end"], alpha=0.30, color="#d73027", label="Noise")

    for seg in row["distortion_spans"]:
        ax.axvspan(seg["start"], seg["end"], alpha=0.28, color="#fdae61", label="Distortion")

    for seg in row["pause_spans"]:
        ax.axvspan(seg["start"], seg["end"], alpha=0.28, color="#4575b4", label="Unnatural Pause")

    ax.plot(times, audio, linewidth=0.65, color="#111111")
    ax.set_title(f"[{row["group"]}] {row["id"]} | MOS: {row["Overall quality"]}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")

    handles, labels = ax.get_legend_handles_labels()
    legend_map = dict(zip(labels, handles))
    if legend_map:
        ax.legend(legend_map.values(), legend_map.keys(), loc="upper right")

    plt.tight_layout()
    plt.show()

    print(f"Noise: {row["Noise Description"]}")
    print(f"Distortion: {row["Distortion description"]}")
    print(f"Pause: {row["Unnatural pause"]}")
    display(Audio(filename=str(wav_path)))
